# CT3 Four-Iteration 199-Case Baseline

## Goal

Train interpretable and nonlinear baseline models using the same frozen
case splits, the same complete elements, the same selected feature set,
and the same upper-tail weights that will be used by symbolic regression.

The baseline does not create the symbolic formula.  It establishes how
much predictive accuracy is available from the current inputs and gives
PySR an honest comparison target.  The final 50 cases remain locked.


## 1. Setup and locked inputs


In [1]:
from pathlib import Path
import gc
import json
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
PACKAGE_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
sys.path.insert(0, str(PACKAGE_ROOT / "src"))

from ct3_common import *

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
warnings.filterwarnings("ignore")

CASE_DIR_OVERRIDE = None
PATHS = build_paths(PACKAGE_ROOT, CASE_DIR_OVERRIDE)
CASE_FILES, CASE_INVENTORY = discover_case_files(PATHS.case_dir)
CASE_PATH_BY_ID = case_path_lookup(CASE_INVENTORY)
SPLIT_MANIFEST = load_frozen_manifest(PATHS.manifest_path, CASE_INVENTORY)
DEVELOPMENT_CASE_IDS = development_case_ids(SPLIT_MANIFEST)
FINAL_CASE_IDS = final_test_case_ids(SPLIT_MANIFEST)

print("Package root:", PATHS.package_root)
print("Case directory:", PATHS.case_dir)
print("Cases:", len(CASE_INVENTORY))
print("Development cases:", len(DEVELOPMENT_CASE_IDS))
print("Locked final-test cases:", len(FINAL_CASE_IDS))
print("Random seed:", RANDOM_SEED)

try:
    from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
except Exception as exc:
    raise ImportError("scikit-learn is required for baseline training") from exc

OUTPUT_DIR = PATHS.output_root / "01_baseline_199cases"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_SELECTION_PATH = PATHS.output_root / "00_qc_sensitivity_ablation" / "locked_feature_set.csv"
FEATURE_SET_NAME, MODEL_FEATURES = feature_set_from_selection(FEATURE_SELECTION_PATH)
RUN_BASELINE_TRAINING = True

print("Locked feature set:", FEATURE_SET_NAME)
print("Features:", MODEL_FEATURES)


Package root: /Users/novwin/Documents/University/University of Manchester/毕设项目/Nuclear graphite/NotebookCT3
Case directory: /Users/novwin/Documents/University/University of Manchester/毕设项目/Nuclear graphite/NotebookCT3/ Case Test Data/FE_Results_Cases_All
Cases: 199
Development cases: 149
Locked final-test cases: 50
Random seed: 42
Locked feature set: full_periodic_coordinates
Features: ['fluence_rate', 'temperature', 'weight_loss_rate', 'rho', 'theta_sin', 'theta_cos', 'z']


## 2. Model definitions

Ridge is a linear reference.  HistGradientBoosting tests smooth nonlinear
partitioning.  ExtraTrees tests flexible interactions and local spatial
structure.  All three use the same case-level split and upper-tail sample
weights, so their metrics are directly comparable to the weighted PySR
objective.  Model selection is validation-only.


In [2]:
def make_models():
    return {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            loss="squared_error", learning_rate=0.05, max_iter=180,
            max_leaf_nodes=63, min_samples_leaf=256, l2_regularization=1.0,
            early_stopping=False, random_state=RANDOM_SEED,
        ),
        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=64, max_depth=24, min_samples_leaf=256,
            max_features=1.0, bootstrap=False, n_jobs=4, # -1 / 4
            random_state=RANDOM_SEED,
        ),
    }

model_parameters = pd.DataFrame([
    {"model": name, "parameters_json": json.dumps(model.get_params(), default=str, sort_keys=True)}
    for name, model in make_models().items()
])
model_parameters.to_csv(OUTPUT_DIR / "baseline_model_parameters.csv", index=False)
display(model_parameters)


,model,parameters_json
0,Ridge,"{""memory"": null, ""ridge"": ""Ridge()"", ""ridge__a..."
1,HistGradientBoosting,"{""categorical_features"": ""warn"", ""early_stoppi..."
2,ExtraTrees,"{""bootstrap"": false, ""ccp_alpha"": 0.0, ""criter..."


## 3. Run four complete case-level iterations


In [3]:
def evaluate_model_cases(model, case_ids, iteration, model_name, split):
    rows = []
    for case_id in case_ids:
        payload, _ = load_case_payload(case_id, CASE_PATH_BY_ID, MODEL_FEATURES)
        predicted = model.predict(payload["X"])
        rows.append({
            "iteration": iteration, "model": model_name, "split": split,
            "case_id": case_id, **evaluate_prediction_arrays(payload["y"], predicted),
        })
        del payload, predicted
        gc.collect()
    return rows

case_metric_records = []
timing_records = []

if RUN_BASELINE_TRAINING:
    for iteration in range(1, 5):
        train_ids = cases_for_role(SPLIT_MANIFEST, iteration, "train")
        validation_ids = cases_for_role(SPLIT_MANIFEST, iteration, "validation")
        internal_ids = cases_for_role(SPLIT_MANIFEST, iteration, "internal_test")
        assert_no_final_cases(train_ids + validation_ids + internal_ids, SPLIT_MANIFEST)

        assembly_start = time.perf_counter()
        bundle = assemble_full_training_arrays(
            train_ids, CASE_PATH_BY_ID, MODEL_FEATURES,
            scale=False, use_tail_weights=True,
        )
        assembly_seconds = time.perf_counter() - assembly_start
        bundle["audit"].assign(iteration=iteration).to_csv(
            OUTPUT_DIR / f"training_weight_audit_iteration_{iteration}.csv", index=False
        )
        print(f"Iteration {iteration}: {len(train_ids)} cases / {len(bundle['y'])} elements")

        for model_name, model in make_models().items():
            fit_start = time.perf_counter()
            if model_name == "Ridge":
                model.fit(bundle["X"], bundle["y"], ridge__sample_weight=bundle["weights"])
            else:
                model.fit(bundle["X"], bundle["y"], sample_weight=bundle["weights"])
            fit_seconds = time.perf_counter() - fit_start

            for split_name, ids in {
                "train": train_ids,
                "validation": validation_ids,
                "internal_test": internal_ids,
            }.items():
                case_metric_records.extend(evaluate_model_cases(
                    model, ids, iteration, model_name, split_name
                ))
            timing_records.append({
                "iteration": iteration, "model": model_name,
                "n_train_cases": len(train_ids), "n_train_elements": len(bundle["y"]),
                "assembly_seconds_shared_by_iteration": assembly_seconds,
                "fit_seconds": fit_seconds,
            })
            pd.DataFrame(case_metric_records).to_csv(
                OUTPUT_DIR / "baseline_case_metrics.csv", index=False
            )
            pd.DataFrame(timing_records).to_csv(
                OUTPUT_DIR / "baseline_timing.csv", index=False
            )
            del model
            gc.collect()
        del bundle
        gc.collect()

    case_metrics = pd.DataFrame(case_metric_records)
    split_metrics = aggregate_case_metrics(case_metrics, ["iteration", "model", "split"])
    split_metrics.to_csv(OUTPUT_DIR / "baseline_split_metrics.csv", index=False)

    metric_columns = [metric for metric, _, _ in SELECTION_METRICS]
    validation_average = (
        split_metrics[split_metrics["split"] == "validation"]
        .groupby("model", observed=True)[metric_columns].mean().reset_index()
    )
    validation_ranking = add_engineering_selection_score(validation_average)
    validation_ranking = validation_ranking.sort_values([
        "engineering_selection_score", "macro_rmse", "model"
    ]).reset_index(drop=True)
    validation_ranking["validation_rank"] = np.arange(1, len(validation_ranking) + 1)
    validation_ranking.to_csv(OUTPUT_DIR / "baseline_validation_ranking.csv", index=False)

    stability = split_metrics.groupby(["model", "split"])[metric_columns].agg(["mean", "std", "min", "max"])
    stability.columns = [f"{metric}_{stat}" for metric, stat in stability.columns]
    stability.reset_index().to_csv(OUTPUT_DIR / "baseline_stability_summary.csv", index=False)
    pd.DataFrame([{
        "feature_set": FEATURE_SET_NAME,
        "features_json": json.dumps(MODEL_FEATURES),
        "manifest_path": str(PATHS.manifest_path),
        "random_seed": RANDOM_SEED,
        "tail_weights_json": json.dumps(TAIL_WEIGHT_LEVELS),
        "final_test_evaluated": False,
    }]).to_csv(OUTPUT_DIR / "baseline_run_contract.csv", index=False)
    display(validation_ranking)
    display(split_metrics)
else:
    print("Baseline training was skipped.")


Iteration 1: 119 cases / 47642840 elements
Iteration 2: 119 cases / 47642840 elements
Iteration 3: 119 cases / 47642840 elements
Iteration 4: 119 cases / 47642840 elements


,model,macro_rmse,worst_case_rmse,mean_top5_actual_rmse,mean_p95_relative_error,mean_p99_relative_error,mean_p99_underprediction_fraction,mean_top5pct_hotspot_overlap,mean_top1pct_hotspot_overlap,mean_top1_recall_in_predicted_top5,macro_rmse_selection_rank,worst_case_rmse_selection_rank,mean_top5_actual_rmse_selection_rank,mean_p95_relative_error_selection_rank,mean_p99_relative_error_selection_rank,mean_p99_underprediction_fraction_selection_rank,mean_top5pct_hotspot_overlap_selection_rank,mean_top1pct_hotspot_overlap_selection_rank,mean_top1_recall_in_predicted_top5_selection_rank,engineering_selection_score,validation_rank
0,ExtraTrees,0.822669,1.959568,1.634813,0.111876,0.114307,0.053259,0.849241,0.839269,0.995971,0.05,0.033333,0.033333,0.05,0.066667,0.066667,0.026667,0.026667,0.013333,0.366667,1
1,HistGradientBoosting,0.961502,2.047953,1.791274,0.113963,0.116790,0.046946,0.808254,0.809732,0.990897,0.10,0.066667,0.066667,0.10,0.133333,0.033333,0.053333,0.053333,0.026667,0.633333,2
2,Ridge,2.955917,4.085838,5.414212,0.177301,0.282459,0.244475,0.248560,0.305203,0.567732,0.15,0.100000,0.100000,0.15,0.200000,0.100000,0.080000,0.080000,0.040000,1.000000,3


,iteration,model,split,n_cases,n_elements_evaluated,micro_mae,micro_rmse,micro_r2,macro_mae,macro_rmse,macro_r2,worst_case_rmse,mean_top5_actual_rmse,mean_top5_actual_bias,mean_p95_relative_error,mean_p95_underprediction_fraction,mean_p99_relative_error,mean_p99_underprediction_fraction,mean_top5pct_hotspot_overlap,mean_top1pct_hotspot_overlap,mean_top1_recall_in_predicted_top5,max_prediction_abs_max_ratio
0,1,ExtraTrees,internal_test,15,6005400,0.433872,0.639736,0.951901,0.433872,0.616200,0.947549,0.948536,1.385052,-0.742639,0.053955,0.042378,0.066227,0.059561,0.871629,0.885115,0.999983,1.093376
1,1,ExtraTrees,train,119,47642840,0.377712,0.561429,0.959708,0.377712,0.544027,0.946216,1.037614,1.163753,-0.237473,0.063847,0.016155,0.064887,0.037169,0.881846,0.882491,0.999994,1.472018
2,1,ExtraTrees,validation,15,6005400,0.710362,1.085189,0.869306,0.710362,0.949289,0.524752,2.283630,2.108518,-0.139570,0.204177,0.045224,0.208459,0.063269,0.843947,0.833949,0.996104,2.130585
3,1,HistGradientBoosting,internal_test,15,6005400,0.586047,0.819296,0.921112,0.586047,0.800716,0.913877,1.170341,1.516614,-0.818555,0.051335,0.040595,0.062207,0.062207,0.825354,0.855378,1.000000,1.100153
4,1,HistGradientBoosting,train,119,47642840,0.552700,0.763257,0.925532,0.552700,0.745325,0.904104,1.291852,1.320347,-0.208653,0.072275,0.016945,0.063715,0.033641,0.834542,0.856289,0.999291,1.554887
5,1,HistGradientBoosting,validation,15,6005400,0.848205,1.236116,0.830424,0.848205,1.112731,0.425652,2.534884,2.190278,0.093564,0.204822,0.035049,0.207349,0.042220,0.792717,0.804895,0.988761,2.229280
6,1,Ridge,internal_test,15,6005400,2.005899,2.858123,0.039954,2.005899,2.849111,-0.047608,3.323452,6.264598,-5.489431,0.165774,0.165774,0.309299,0.309299,0.248190,0.290393,0.559607,0.587949
7,1,Ridge,train,119,47642840,2.107080,2.903078,-0.077327,2.107080,2.884398,-0.323389,3.935175,5.332034,-4.411691,0.146655,0.098129,0.250049,0.240606,0.255172,0.312263,0.565882,0.813407
8,1,Ridge,validation,15,6005400,2.396000,3.253649,-0.174862,2.396000,3.176508,-1.993653,5.600438,5.692227,-4.222627,0.320682,0.132701,0.370482,0.243416,0.254568,0.336180,0.585048,1.262451
9,2,ExtraTrees,internal_test,15,6005400,0.535552,0.753845,0.927512,0.535552,0.703295,0.917642,1.278475,1.371582,-0.472689,0.059922,0.024269,0.069208,0.050417,0.865411,0.848701,0.998685,1.191899


## Takeaways

`baseline_validation_ranking.csv` is the comparison interface consumed
by the symbolic-regression notebook.  The symbolic search can technically
run without a baseline, but the formal CT3 workflow requires this baseline
to be completed first so formula accuracy can be judged against a known
nonlinear reference under identical splits and metrics.
